# Notebook 04 — My Report Runs Out of Memory
## Right-Sizing Your Warehouse to Avoid Spilling

**The scenario**: Your quarterly fraud summary runs fine on small date ranges, but when you extend it to all of 2024 across 50M accounts, it suddenly takes 10x longer. The query profile shows "bytes spilled to local/remote storage."

**What's happening**: The warehouse doesn't have enough memory to hold all the intermediate results (millions of GROUP BY groups). It "spills" data to disk — like your laptop swapping to disk when RAM fills up.

**The fix**: Use a larger warehouse for heavy aggregations. Bigger warehouse = more memory = no spilling.

| Warehouse | Memory | Spilling? | Cost/hr |
|-----------|--------|-----------|--------|
| XS | 8 GB | Spills on large GROUP BY | 1 credit |
| M | 64 GB | Usually fits in memory | 4 credits |
| L | 128 GB | Definitely fits | 8 credits |

In [ ]:
USE ROLE SYSADMIN;
USE SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Worst Case: XS Warehouse — Not Enough Memory

Per-account monthly fraud analysis (millions of groups → spills to disk).

In [ ]:
USE WAREHOUSE HOL_XS;

-- Heavy aggregation: per-account monthly totals (millions of groups)
SELECT
    account_id,
    DATE_TRUNC('month', transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount,
    SUM(CASE WHEN fraud_flag THEN amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS
WHERE transaction_date >= '2024-01-01'
GROUP BY 1, 2
ORDER BY fraud_losses DESC
LIMIT 100;

---
## Best Case: M Warehouse — Enough Memory

In [ ]:
USE WAREHOUSE HOL_M;

-- Same query, bigger warehouse — fits in memory
SELECT
    account_id,
    DATE_TRUNC('month', transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_amount,
    AVG(amount) AS avg_amount,
    SUM(CASE WHEN fraud_flag THEN amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS
WHERE transaction_date >= '2024-01-01'
GROUP BY 1, 2
ORDER BY fraud_losses DESC
LIMIT 100;

---
## Compare Performance

In [ ]:
SELECT
    warehouse_name,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -10, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 10
))
WHERE query_text ILIKE '%fraud_losses%'
  AND query_text ILIKE '%DATE_TRUNC%'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 2;

---
## How to Check for Spilling (Query Profile)

After running a slow query, check the **Query Profile** in Snowsight:
1. Click on the query in History
2. Open the Query Profile tab
3. Look for **Bytes Spilled to Local Storage** or **Bytes Spilled to Remote Storage**

If you see spilling → the warehouse is too small for that query.

---
## Key Takeaway

| Metric | XS Warehouse | M Warehouse | Why |
|--------|-------------|-------------|-----|
| Elapsed Time | 30-60 sec | 5-10 sec | No disk spilling |
| Spilling | Yes (local/remote) | No | 8x more memory |
| Cost | 1 credit/hr × long time | 4 credits/hr × short time | Often cheaper! |

**The counterintuitive truth**: A bigger warehouse can be *cheaper* because it finishes faster. 1 credit/hr × 60 seconds = 0.017 credits. 4 credits/hr × 10 seconds = 0.011 credits.

**When to upsize your warehouse:**
- Query Profile shows spilling
- Your GROUP BY has millions of groups
- Large JOINs between big tables
- Complex window functions on large datasets

**Next** → [Notebook 05 — Query Tuning](./Notebook_05_Query_Tuning.ipynb) (free optimizations in YOUR SQL)